# 01 – FHIR Patient Ingestion

Fetch Patient resources from the public FHIR API, preserve the original paginated JSON responses in the Lakehouse, and load versioned records into the Bronze Delta table. The watermark advances only after raw and Bronze data are saved successfully.

## Cell 1 – Create the watermark table

Run this cell once before the configuration cell. It creates the shared watermark table if it does not already exist.

In [1]:
# CREATE WATERMARK TABLE
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.fhir_watermark (
    resource_type STRING,
    last_successful_timestamp TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
""")

print("Watermark table is ready.")

StatementMeta(, 06edbe6d-49c3-4eb6-aed7-87e77295edb4, 3, Finished, Available, Finished, False)

Watermark table is ready.


## Cell 2 – Imports and configuration

The first run uses a three-day lookback. Later runs read the last successful watermark and include a one-hour overlap.

In [2]:
import requests
from datetime import datetime, timezone, timedelta

# CONFIGURATION
FHIR_BASE_URL = "https://hapi.fhir.org/baseR4"
RESOURCE_TYPE = "Patient"
PAGE_SIZE = 100

# Capture the end time for this extraction run
extraction_timestamp = datetime.now(timezone.utc)

# Read the last successful watermark for this resource
watermark_row = spark.sql(f"""
    SELECT last_successful_timestamp
    FROM bronze.fhir_watermark
    WHERE resource_type = '{RESOURCE_TYPE}'
""").first()

# First run: extract the last three days.
# Later runs: start one hour before the previous successful watermark.
if watermark_row is None or watermark_row["last_successful_timestamp"] is None:
    start_date = extraction_timestamp - timedelta(days=3)
else:
    last_successful = watermark_row["last_successful_timestamp"].replace(
        tzinfo=timezone.utc
    )
    start_date = last_successful - timedelta(hours=1)

start_time = start_date.strftime("%Y-%m-%dT%H:%M:%SZ")
end_time = extraction_timestamp.strftime("%Y-%m-%dT%H:%M:%SZ")

# API endpoint and query parameters
api_url = f"{FHIR_BASE_URL}/{RESOURCE_TYPE}"

params = [
    ("_count", str(PAGE_SIZE)),
    ("_lastUpdated", f"ge{start_time}"),
    ("_lastUpdated", f"lt{end_time}"),
    ("_format", "json")
]

headers = {
    "Accept": "application/fhir+json"
}

print("Resource:", RESOURCE_TYPE)
print("Start date:", start_time)
print("End date:", end_time)
print("Extraction timestamp:", extraction_timestamp.isoformat())

StatementMeta(, 06edbe6d-49c3-4eb6-aed7-87e77295edb4, 4, Finished, Available, Finished, False)

Resource: Patient
Start date: 2026-11-18T18:37:38Z
End date: 2026-11-18T19:58:10Z
Extraction timestamp: 2026-11-18T19:58:10.818635+00:00


## Cell 3 – Fetch paginated API responses and save raw JSON

Each original FHIR Bundle response is saved without modifying its JSON text.

In [3]:

# PAGINATION AND RAW JSON STORAGE

import uuid

extraction_date = extraction_timestamp.strftime("%Y-%m-%d")

raw_folder = (
    f"Files/raw/{RESOURCE_TYPE}/"
    f"extraction_date={extraction_date}"
)

notebookutils.fs.mkdirs(raw_folder)

# Generate a unique ID for this notebook execution
run_id = uuid.uuid4().hex

next_url = api_url
next_params = params
page_number = 1
total_records = 0

while next_url:
    print(f"Fetching page {page_number}...")

    response = requests.get(
        next_url,
        headers=headers,
        params=next_params,
        timeout=120
    )
    response.raise_for_status()

    # Preserve the original API response
    raw_json = response.text
    bundle = response.json()

    # Unique filename for every execution
    file_path = (
        f"{raw_folder}/"
        f"page_{page_number:04d}_{run_id}.json"
    )

    notebookutils.fs.put(
        file_path,
        raw_json,
        False
    )

    records_in_page = len(bundle.get("entry", []))
    total_records += records_in_page

    print(f"Page {page_number}: {records_in_page} records saved")

    next_link = next(
        (
            link["url"]
            for link in bundle.get("link", [])
            if link.get("relation") == "next"
        ),
        None
    )

    # The next URL already contains its own query parameters
    next_url = next_link
    next_params = None
    page_number += 1

total_pages = page_number - 1

print("--------------------------------")
print("PATIENT INGESTION COMPLETED")
print("Total pages:", total_pages)
print("Total records:", total_records)
print("Raw folder:", raw_folder)
print("Run ID:", run_id)


StatementMeta(, 06edbe6d-49c3-4eb6-aed7-87e77295edb4, 5, Finished, Available, Finished, False)

Fetching page 1...
Page 1: 31 records saved
--------------------------------
PATIENT INGESTION COMPLETED
Total pages: 1
Total records: 31
Raw folder: Files/raw/Patient/extraction_date=2026-11-18


## Cell 4 – Read the saved raw JSON Bundles

In [4]:
from pyspark.sql.functions import (
    col,
    explode,
    to_json,
    current_timestamp,
    lit,
    input_file_name
)

raw_df = (
    spark.read
    .option("multiLine", "true")
    .json(raw_folder)
)

print("Raw Bundle files loaded.")
raw_df.printSchema()

StatementMeta(, 06edbe6d-49c3-4eb6-aed7-87e77295edb4, 6, Finished, Available, Finished, False)

Raw Bundle files loaded.
root
 |-- entry: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- fullUrl: string (nullable = true)
 |    |    |-- resource: struct (nullable = true)
 |    |    |    |-- _active: struct (nullable = true)
 |    |    |    |    |-- extension: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |    |    |-- valueString: string (nullable = true)
 |    |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- _birthDate: struct (nullable = true)
 |    |    |    |    |-- extension: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |    |    |-- valueDateTime: string (nullable = true)
 |    |    |    |-- active: boolean (nullable = true)
 |    |    |    |-- address: array

## Cell 5 – Extract individual Patient resources

In [5]:
patient_df = (
    raw_df
    .select(explode(col("entry")).alias("entry"))
    .select(col("entry.resource").alias("resource"))
)

print("Individual Patient resources extracted.")
print("Patient records:", patient_df.count())

patient_df.printSchema()

StatementMeta(, 06edbe6d-49c3-4eb6-aed7-87e77295edb4, 7, Finished, Available, Finished, False)

Individual Patient resources extracted.
Patient records: 1363
root
 |-- resource: struct (nullable = true)
 |    |-- _active: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueString: string (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |-- _birthDate: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueDateTime: string (nullable = true)
 |    |-- active: boolean (nullable = true)
 |    |-- address: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- city: string (nullable = true)
 |    |    |    |-- country: string (nullable = true)
 |    |    |    |-- district: string (nullable = true)
 |  

## Cell 6 – Prepare the Bronze DataFrame

The complete FHIR resource is retained in `resource_json`. The Bronze table uses a consistent seven-column schema.

In [6]:
extraction_ts = extraction_timestamp.isoformat()
api_details = f"{api_url} | Parameters: {params}"

bronze_patient_df = (
    patient_df
    .select(
        col("resource.*"),
        to_json(col("resource")).alias("resource_json")
    )
    .withColumn(
        "extraction_timestamp",
        lit(extraction_ts).cast("timestamp")
    )
    .withColumn(
        "api_url_or_params",
        lit(api_details)
    )
    .withColumn(
        "load_timestamp",
        current_timestamp()
    )
    .withColumn(
        "source_file",
        input_file_name()
    )
)

bronze_patient_df.printSchema()
bronze_patient_df.show(5, truncate=False)

StatementMeta(, 06edbe6d-49c3-4eb6-aed7-87e77295edb4, 8, Finished, Available, Finished, False)

root
 |-- _active: struct (nullable = true)
 |    |-- extension: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- url: string (nullable = true)
 |    |    |    |-- valueString: string (nullable = true)
 |    |-- id: string (nullable = true)
 |-- _birthDate: struct (nullable = true)
 |    |-- extension: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- url: string (nullable = true)
 |    |    |    |-- valueDateTime: string (nullable = true)
 |-- active: boolean (nullable = true)
 |-- address: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |    |-- district: string (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- extension: array (nullable = true)
 |    |    |    |    | 

## Cell 7 – Merge Bronze records and update the watermark

This avoids replacing the existing Bronze table. A new row is inserted when the Patient ID and original JSON combination has not already been stored. The watermark is updated only after the raw and Bronze steps complete.

In [7]:
from delta.tables import DeltaTable

bronze_patient_final = (
    bronze_patient_df
    .select(
        "id",
        "resourceType",
        "resource_json",
        "extraction_timestamp",
        "api_url_or_params",
        "load_timestamp",
        "source_file"
    )
    .dropDuplicates(["id", "resource_json"])
)

# Create the Bronze table on the first run.
# On later runs, retain existing records and insert unseen resource versions.
if not spark.catalog.tableExists("bronze.patient"):
    (
        bronze_patient_final.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("bronze.patient")
    )
else:
    bronze_table = DeltaTable.forName(spark, "bronze.patient")

    (
        bronze_table.alias("target")
        .merge(
            bronze_patient_final.alias("source"),
            """
            target.id = source.id
            AND target.resource_json = source.resource_json
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

print("Bronze Patient Delta table saved successfully.")
print("Total Bronze records:", spark.table("bronze.patient").count())

# Advance the watermark only after raw and Bronze data have been saved.
watermark_update_df = spark.createDataFrame(
    [(RESOURCE_TYPE, extraction_timestamp.replace(tzinfo=None))],
    ["resource_type", "last_successful_timestamp"]
)

watermark_table = DeltaTable.forName(spark, "bronze.fhir_watermark")

(
    watermark_table.alias("target")
    .merge(
        watermark_update_df.alias("source"),
        "target.resource_type = source.resource_type"
    )
    .whenMatchedUpdate(
        condition="""
            target.last_successful_timestamp IS NULL
            OR source.last_successful_timestamp > target.last_successful_timestamp
        """,
        set={
            "last_successful_timestamp": "source.last_successful_timestamp",
            "updated_at": "current_timestamp()"
        }
    )
    .whenNotMatchedInsert(
        values={
            "resource_type": "source.resource_type",
            "last_successful_timestamp": "source.last_successful_timestamp",
            "updated_at": "current_timestamp()"
        }
    )
    .execute()
)

print("Patient watermark updated:", extraction_timestamp.isoformat())

StatementMeta(, 06edbe6d-49c3-4eb6-aed7-87e77295edb4, 9, Finished, Available, Finished, False)

Bronze Patient Delta table saved successfully.
Total Bronze records: 1485
Patient watermark updated: 2026-11-18T19:58:10.818635+00:00


## Cell 8 – Check the Patient watermark

Run this after the Bronze merge cell succeeds.

In [8]:
spark.sql("""
    SELECT *
    FROM bronze.fhir_watermark
    WHERE resource_type = 'Patient'
""").show(truncate=False)

StatementMeta(, 06edbe6d-49c3-4eb6-aed7-87e77295edb4, 10, Finished, Available, Finished, False)

+-------------+--------------------------+--------------------------+
|resource_type|last_successful_timestamp |updated_at                |
+-------------+--------------------------+--------------------------+
|Patient      |2026-11-18 19:58:10.818635|2026-11-18 19:58:34.919831|
+-------------+--------------------------+--------------------------+

